# Classification Into Employer-Branding And Product-Technology Videos

This notebook is the data extraction step for the project. It connects to the MongoDB `youtube.videos` collection, retrieves tag frequencies, classifies videos into employer-branding and product-technology groups with MongoDB aggregation pipelines, and stores the exported datasets in the `data` folder for later notebooks.

This notebook is where MongoDB is used most heavily because the classification logic itself is a database task rather than a pandas task.


## Workflow

The notebook performs four tasks:

1. Connect to MongoDB in read-only mode.
2. Retrieve tag frequencies from `snippet.tags`.
3. Build the employer-branding and product-technology datasets with MongoDB aggregations.
4. Export the resulting datasets to the `data` folder for downstream validation and analysis.


In [5]:
import json
import os
from pathlib import Path
from getpass import getpass

import certifi
import pandas as pd

from pymongo import MongoClient

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

if not os.getenv("MONGODB_URI"):
    os.environ["MONGODB_URI"] = getpass("Enter MongoDB URI: ")



## Connect To MongoDB

Set the connection string in the environment variable `MONGODB_URI` before running the notebook.


In [6]:
MONGODB_URI = os.getenv("MONGODB_URI")

if not MONGODB_URI:
    raise ValueError("Set the MONGODB_URI environment variable before running this notebook.")

client = MongoClient(
    MONGODB_URI,
    tlsCAFile=certifi.where(),
    serverSelectionTimeoutMS=30000,
)
db = client["youtube"]
videos = db["videos"]

db.list_collection_names()


['channels',
 'subscriptions',
 'extendedStatistics',
 'videoCommentThreads',
 'companies',
 'captions',
 'videoCommentThreadReplies',
 'playlistItems',
 'videos',
 'videoCategories',
 'playlist_items',
 'activities',
 'playlists',
 'channelCommentThreadReplies',
 'channelCommentThreads']

## Tag Extraction With MongoDB

This aggregation uses nested array access, `$unwind`, `$group`, `$sort` and a string operator to retrieve the most frequent tags in the dataset. The output is stored in `youtube_tag_frequencies.json` so the classification logic is reproducible.


In [7]:
tag_frequency_pipeline = [
    {"$match": {"snippet.tags": {"$exists": True, "$ne": []}}},
    {"$unwind": "$snippet.tags"},
    {
        "$group": {
            "_id": {"$toLower": "$snippet.tags"},
            "count": {"$sum": 1},
        }
    },
    {"$sort": {"count": -1, "_id": 1}},
]

tag_frequencies = list(videos.aggregate(tag_frequency_pipeline, allowDiskUse=True))

with open("../youtube_tag_frequencies.json", "w", encoding="utf-8") as handle:
    json.dump(tag_frequencies, handle, ensure_ascii=False, indent=2)

pd.DataFrame(tag_frequencies).head(20)


,_id,count
0,credit suisse,2044
1,nestle,1358
2,cs,1288
3,cbbcs,1264
4,warrants,1264
5,熊證,1264
6,牛熊証,1264
7,牛熊證,1264
8,牛證,1264
9,瑞信,1264


## Classification Rules

The following regex lists define the two content groups. The queries use nested tag matching with `$elemMatch` and regex to identify relevant videos directly in MongoDB.


In [8]:
employee_regex = (
    "employee|employees|career|careers|job|jobs|work|team|hr|human resources|"
    "internship|internships|recruiting|karriere|mitarbeiter|ausbildung|"
    "lehrstelle|arbeit|emploi|lavoro|trabajo|employer branding"
)

product_regex = (
    "product|products|service|services|innovation|technology|software|app|"
    "application|tutorial|how to|demo|installation|commercial|advertisement|"
    "advertising|spot|tv spot|werbung|brand|machine|system|solution"
)

employee_match = {
    "snippet.tags": {
        "$elemMatch": {
            "$regex": employee_regex,
            "$options": "i",
        }
    }
}

product_match = {
    "snippet.tags": {
        "$elemMatch": {
            "$regex": product_regex,
            "$options": "i",
        }
    }
}

employee_match, product_match


({'snippet.tags': {'$elemMatch': {'$regex': 'employee|employees|career|careers|job|jobs|work|team|hr|human resources|internship|internships|recruiting|karriere|mitarbeiter|ausbildung|lehrstelle|arbeit|emploi|lavoro|trabajo|employer branding',
    '$options': 'i'}}},
 {'snippet.tags': {'$elemMatch': {'$regex': 'product|products|service|services|innovation|technology|software|app|application|tutorial|how to|demo|installation|commercial|advertisement|advertising|spot|tv spot|werbung|brand|machine|system|solution',
    '$options': 'i'}}})

## Export Employer-Branding Videos

This aggregation uses `$match` and `$project` to create an analysis-ready subset while keeping the structure close to the exported files used later in the project.


In [9]:
employee_pipeline = [
    {"$match": employee_match},
    {
        "$project": {
            "_id": 1,
            "company": 1,
            "channelId": 1,
            "title": "$snippet.title",
            "description": "$snippet.description",
            "tags": "$snippet.tags",
            "publishedAt": "$snippet.publishedAt",
            "views": {"$toDouble": "$statistics.viewCount"},
            "likes": {"$toDouble": "$statistics.likeCount"},
            "comments": {"$toDouble": "$statistics.commentCount"},
        }
    },
]

employee_docs = list(videos.aggregate(employee_pipeline, allowDiskUse=True))
employee_df = pd.DataFrame(employee_docs)

employee_df.to_json(DATA_DIR / "employee-based-content.json", orient="records", lines=True)

print("Employer-branding videos exported:", len(employee_df))
employee_df.head()


Employer-branding videos exported: 3120


,_id,channelId,company,title,description,tags,publishedAt,views,likes,comments
0,jiHmzKeqeT8,UCLAKVzA5WEadvys9CKOksQg,Adecco SA,Klåss Clerkx - CEO for One Month 2016 at Adecc...,The #CEO1Month selection process started with ...,"[CEO1MONTH, Adecco Way to Work, bootcamp]",2016-05-25T15:51:58.000Z,156.0000,1.0000,0.0000
1,1p9NjGCEieg,UCrSSdGg-aH_HgCHtsnoonIg,Adecco SA,Adecco Thailand WayToWork 2014,กะเทาะเปลือกเด็กจบใหม่ ครั้งที่ 2 กิจกรรมเพื่อ...,"[Way2WorkTh, AdeccoThailand, WayToWork, HRtwt,...",2014-04-30T11:10:20.000Z,"1,249.0000",5.0000,1.0000
2,bmvCxO3zvsQ,UCrSSdGg-aH_HgCHtsnoonIg,Adecco SA,Adecco Thailand Career Up - Marketing Jobs,How to Introduce Yourself in English for Marke...,"[Marketing, Adecco, Jobs Seeker, Jobs]",2011-09-23T03:24:05.000Z,"4,535.0000",6.0000,3.0000
3,61taXGTki5g,UCLAKVzA5WEadvys9CKOksQg,Adecco SA,Jobs bij TNT: Customer Service medewerkers in ...,"Je bent klantvriendelijk, goed in communicatie...","[Jobs, TNT, Customer Service, Machelen, België]",2016-05-17T11:15:11.000Z,131.0000,1.0000,0.0000
4,W33uEUbULRs,UC7wvKWegeU9FGzEcVhJVoDw,ABB Ltd,Anna – Success begins with you,"Success begins with you because, at ABB, you c...","[ABB Ltd, ABB Careers, Technology, Engineering]",2016-07-18T14:27:22.000Z,379.0000,5.0000,1.0000


## Export Product-Technology Videos

The product-technology extraction follows the same structure as the employer-branding extraction so the two datasets are directly comparable in the later notebooks.


In [10]:
product_pipeline = [
    {"$match": product_match},
    {
        "$project": {
            "_id": 1,
            "company": 1,
            "channelId": 1,
            "title": "$snippet.title",
            "description": "$snippet.description",
            "tags": "$snippet.tags",
            "publishedAt": "$snippet.publishedAt",
            "views": {"$toDouble": "$statistics.viewCount"},
            "likes": {"$toDouble": "$statistics.likeCount"},
            "comments": {"$toDouble": "$statistics.commentCount"},
        }
    },
]

product_docs = list(videos.aggregate(product_pipeline, allowDiskUse=True))
product_df = pd.DataFrame(product_docs)

product_df.to_json(DATA_DIR / "product-based-content.json", orient="records", lines=True)

print("Product-technology videos exported:", len(product_df))
product_df.head()


Product-technology videos exported: 4605


,_id,channelId,company,title,description,tags,publishedAt,views,likes,comments
0,EvjO6DIurLk,UCXyz4vATE49KlC_7vUY-bdw,Cembra Money Bank AG,Cembra Money Bank Famiglia con divano Italiano,Rebranding: Campagna Autunno 2013,"[Cembra Money Bank, Familie, Sofa, GE Money Ba...",2013-11-11T10:26:04.000Z,"27,387.0000",0.0000,0.0000
1,jMlOv0Z5GpE,UCJ8IXb4xgt85gSKMpxXfpVQ,Bossard Holding AG,So funktioniert die Wärme-Einbettung eines Tap...,In warmem Kunststoff feste Verbindungen schaff...,"[kvt, fastening, kvt-fastening, Verbindungsele...",2015-06-22T07:47:56.000Z,293.0000,NaN,NaN
2,61taXGTki5g,UCLAKVzA5WEadvys9CKOksQg,Adecco SA,Jobs bij TNT: Customer Service medewerkers in ...,"Je bent klantvriendelijk, goed in communicatie...","[Jobs, TNT, Customer Service, Machelen, België]",2016-05-17T11:15:11.000Z,131.0000,1.0000,0.0000
3,W33uEUbULRs,UC7wvKWegeU9FGzEcVhJVoDw,ABB Ltd,Anna – Success begins with you,"Success begins with you because, at ABB, you c...","[ABB Ltd, ABB Careers, Technology, Engineering]",2016-07-18T14:27:22.000Z,379.0000,5.0000,1.0000
4,jOWd1QNgr7c,UCf6fYOaOmIGuLPSGncxELNA,Baloise Holding AG,Baloise Jobcast #4: Personal Branding,Wie präsentiere ich meine Eigenmarke? In diese...,"[bewerben, auftritt, image, baloise, podcast, ...",2016-05-02T14:47:55.000Z,47.0000,1.0000,0.0000


## Quick MongoDB Checks

These short checks document the size of the extracted datasets with MongoDB operations rather than pandas where possible.


In [11]:
employee_count = videos.count_documents(employee_match)
product_count = videos.count_documents(product_match)
employee_companies = videos.distinct("company", employee_match)
product_companies = videos.distinct("company", product_match)

pd.DataFrame(
    {
        "dataset": ["employer_branding", "product_technology"],
        "video_count": [employee_count, product_count],
        "unique_companies": [len(employee_companies), len(product_companies)],
    }
)


,dataset,video_count,unique_companies
0,employer_branding,3120,94
1,product_technology,4605,110


The notebook writes three project files:

- `youtube_tag_frequencies.json`
- `data/employee-based-content.json`
- `data/product-based-content.json`

These files are then used by the validation notebook and the final audience-conversation notebook.
